**Note (2026-08-30):** this notebook's logic has been migrated into the real package modules — `ml/data/md17.py`, `ml/data/preprocessing.py`, `ml/data/datasets.py` — which is where training/eval code should import from now. Kept here as the original dev record. The zip-loading bug below (`__MACOSX` AppleDouble junk entries breaking `load_split_zip`, causing the 4 zip-based sources to fail) is fixed in `ml/data/md17.py`; running `python -m ml.data.preprocessing` gets 9/9 sources instead of 5/9.

Import (standard library)

In [42]:
from __future__ import annotations

import io
import re
import subprocess
import sys
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

Install  & import ML libraries

In [43]:
try:
    import torch_geometric
except ImportError:
    print("Installing torch_geometric ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "torch_geometric"], check=True)

import numpy as np
import torch
from torch_geometric.data import Data, Dataset

Clone the repo and setup data-lake paths

In [44]:
REPO_URL = "https://github.com/AlexTanui/HydrogenProductionMaterials_Discovery.git"
REPO_DIR = Path("/content/HydrogenProductionMaterials_Discovery")

if not REPO_DIR.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repo already present at {REPO_DIR}, skipping clone.")

ROOT = REPO_DIR
BRONZE = ROOT / "data" / "bronze" / "md17"
SILVER = ROOT / "data" / "silver" / "md17"
GOLD = ROOT / "data" / "gold" / "md17"

Repo already present at /content/HydrogenProductionMaterials_Discovery, skipping clone.


Run the bronze data and download script

In [45]:
download_script = ROOT / "scripts" / "download_bronze_data.sh"
if download_script.exists():
    print("Running scripts/download_bronze_data.sh ...")
    subprocess.run(["bash", str(download_script)], cwd=str(ROOT), check=True)
else:
    print(f"WARNING: {download_script} not found — skipping download step.")

Running scripts/download_bronze_data.sh ...


Core data model and parsing helpers

In [46]:
REQUIRED_KEYS = ("z", "R", "E", "F")
_PERIODIC_TABLE = {"H": 1, "C": 6, "N": 7, "O": 8, "F": 9, "S": 16, "Cl": 17}


@dataclass
class MD17Sample:
    """One molecule+theory's worth of trajectory data, already validated."""

    molecule: str
    theory: str
    z: np.ndarray
    R: np.ndarray
    E: np.ndarray
    F: np.ndarray

    def __post_init__(self) -> None:
        self.z = np.asarray(self.z, dtype=np.int64).reshape(-1)
        self.R = np.asarray(self.R, dtype=np.float32)
        self.F = np.asarray(self.F, dtype=np.float32)
        self.E = np.asarray(self.E, dtype=np.float32).reshape(-1)

        if self.R.ndim == 2:
            self.R = self.R[None, ...]
        if self.F.ndim == 2:
            self.F = self.F[None, ...]

        n_atoms = self.z.shape[0]
        if self.R.shape[1:] != (n_atoms, 3):
            raise ValueError(f"{self.molecule}: R shape {self.R.shape} inconsistent with {n_atoms} atoms")
        if self.F.shape != self.R.shape:
            raise ValueError(f"{self.molecule}: F shape {self.F.shape} != R shape {self.R.shape}")
        if self.E.shape[0] != self.R.shape[0]:
            raise ValueError(f"{self.molecule}: E has {self.E.shape[0]} entries, R has {self.R.shape[0]} configs")

    @property
    def n_atoms(self) -> int:
        return int(self.z.shape[0])

    @property
    def n_configs(self) -> int:
        return int(self.R.shape[0])


def _infer_theory(filename: str) -> str:
    name = filename.lower()
    if "ccsd_t" in name or "ccsd-t" in name:
        return "ccsd_t"
    if "ccsd" in name:
        return "ccsd"
    return "dft"


def _infer_molecule(filename: str) -> str:
    stem = Path(filename).stem
    stem = re.sub(r"^md1[78]_", "", stem)
    stem = re.sub(r"[_-]?(dft|ccsd_t|ccsd-t|ccsd|train|test)$", "", stem, flags=re.IGNORECASE)
    return stem


def load_npz(path: str | Path, molecule: Optional[str] = None, theory: Optional[str] = None) -> MD17Sample:
    path = Path(path)
    data = np.load(path, allow_pickle=True)
    missing = [k for k in REQUIRED_KEYS if k not in data]
    if missing:
        raise ValueError(f"{path.name}: missing required keys {missing}, has {list(data.keys())}")
    inferred_molecule = str(data["name"]) if "name" in data.files else _infer_molecule(path.name)
    return MD17Sample(
        molecule=molecule or inferred_molecule,
        theory=theory or _infer_theory(path.name),
        z=data["z"], R=data["R"], E=data["E"], F=data["F"],
    )


def _split_xyz_frames(text: str) -> list[str]:
    lines = text.strip().splitlines()
    frames = []
    i = 0
    while i < len(lines):
        if not lines[i].strip():
            i += 1
            continue
        n_atoms = int(lines[i].strip())
        frames.append("\n".join(lines[i : i + 2 + n_atoms]))
        i += 2 + n_atoms
    return frames


def _parse_xyz_frame(text: str) -> tuple[np.ndarray, float, np.ndarray, np.ndarray]:
    lines = [l for l in text.strip().splitlines() if l.strip()]
    n_atoms = int(lines[0].strip())
    comment = lines[1]
    energy_match = re.search(r"[-+]?\d*\.\d+(?:[eE][-+]?\d+)?", comment)
    energy = float(energy_match.group()) if energy_match else float("nan")
    symbols, coords, forces = [], [], []
    for line in lines[2 : 2 + n_atoms]:
        parts = line.split()
        symbols.append(parts[0])
        coords.append([float(x) for x in parts[1:4]])
        forces.append([float(x) for x in parts[4:7]])
    z = np.array([_PERIODIC_TABLE.get(s, 0) for s in symbols], dtype=np.int64)
    return z, energy, np.array(coords, dtype=np.float32), np.array(forces, dtype=np.float32)


def _split_name(member: str) -> Optional[str]:
    lower = member.lower()
    if "train" in lower:
        return "train"
    if "test" in lower:
        return "test"
    return None


def load_split_zip(path: str | Path, molecule: Optional[str] = None, theory: Optional[str] = None) -> dict[str, MD17Sample]:
    path = Path(path)
    molecule = molecule or _infer_molecule(path.name)
    theory = theory or _infer_theory(path.name)

    result: dict[str, MD17Sample] = {}
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
        npz_members = [n for n in names if n.lower().endswith(".npz")]
        xyz_members = [n for n in names if n.lower().endswith(".xyz")]

        if npz_members:
            for member in npz_members:
                split = _split_name(member)
                if split is None:
                    continue
                with zf.open(member) as f:
                    buf = io.BytesIO(f.read())
                    data = np.load(buf, allow_pickle=True)
                    result[split] = MD17Sample(molecule=molecule, theory=theory, z=data["z"], R=data["R"], E=data["E"], F=data["F"])
        elif xyz_members:
            for member in xyz_members:
                split = _split_name(member)
                if split is None:
                    continue
                with zf.open(member) as f:
                    text = f.read().decode("utf-8")
                z = None
                Rs, Es, Fs = [], [], []
                for frame_text in _split_xyz_frames(text):
                    fz, e, r, force = _parse_xyz_frame(frame_text)
                    z = fz if z is None else z
                    Rs.append(r); Es.append(e); Fs.append(force)
                result[split] = MD17Sample(molecule=molecule, theory=theory, z=z, R=np.stack(Rs), E=np.array(Es), F=np.stack(Fs))
        else:
            raise ValueError(f"{path.name}: no .npz or .xyz members found ({names})")

    if "train" not in result or "test" not in result:
        raise ValueError(f"{path.name}: expected both train and test splits, found {list(result.keys())}")
    return result


def build_graph(R_frame: np.ndarray, cutoff: float = 5.0) -> tuple[torch.Tensor, torch.Tensor]:
    pos = torch.as_tensor(R_frame, dtype=torch.float32)
    dist = torch.cdist(pos, pos)
    n_atoms = pos.shape[0]
    mask = (dist <= cutoff) & ~torch.eye(n_atoms, dtype=torch.bool)
    row, col = mask.nonzero(as_tuple=True)
    edge_index = torch.stack([row, col], dim=0)
    edge_attr = dist[row, col].unsqueeze(-1)
    return edge_index, edge_attr


Validation and bronze-silver-gold pipeline

In [47]:
def validate_sample(sample: MD17Sample) -> list[str]:
    problems = []
    if np.isnan(sample.R).any():
        problems.append("NaN values in R")
    if np.isnan(sample.E).any():
        problems.append("NaN values in E")
    if np.isnan(sample.F).any():
        problems.append("NaN values in F")
    if sample.n_atoms == 0:
        problems.append("zero atoms")
    if sample.n_configs == 0:
        problems.append("zero configurations")
    return problems


def bronze_to_silver(bronze_path: str | Path, silver_dir: str | Path,
                      molecule: Optional[str] = None, theory: Optional[str] = None) -> Path:
    bronze_path = Path(bronze_path)
    silver_dir = Path(silver_dir)
    silver_dir.mkdir(parents=True, exist_ok=True)

    if bronze_path.suffix == ".npz":
        sample = load_npz(bronze_path, molecule=molecule, theory=theory)
        problems = validate_sample(sample)
        if problems:
            raise ValueError(f"{bronze_path.name}: failed validation: {problems}")
        literature_test_mask = np.zeros(sample.n_configs, dtype=bool)
        has_literature_split = False

    elif bronze_path.suffix == ".zip":
        splits = load_split_zip(bronze_path, molecule=molecule, theory=theory)
        train, test = splits["train"], splits["test"]
        for name, s in (("train", train), ("test", test)):
            problems = validate_sample(s)
            if problems:
                raise ValueError(f"{bronze_path.name} [{name}]: failed validation: {problems}")
        sample = MD17Sample(
            molecule=train.molecule, theory=train.theory, z=train.z,
            R=np.concatenate([train.R, test.R], axis=0),
            E=np.concatenate([train.E, test.E], axis=0),
            F=np.concatenate([train.F, test.F], axis=0),
        )
        literature_test_mask = np.concatenate([np.zeros(train.n_configs, dtype=bool), np.ones(test.n_configs, dtype=bool)])
        has_literature_split = True
    else:
        raise ValueError(f"Unsupported bronze file type: {bronze_path.suffix}")

    out_path = silver_dir / f"{sample.molecule}_{sample.theory}.npz"
    np.savez_compressed(
        out_path, z=sample.z, R=sample.R, E=sample.E, F=sample.F,
        molecule=sample.molecule, theory=sample.theory,
        has_literature_split=has_literature_split, literature_test_mask=literature_test_mask,
    )
    return out_path


def trajectory_block_split(n_configs: int, train: float = 0.8, val: float = 0.1, test: float = 0.1):
    assert abs(train + val + test - 1.0) < 1e-6, "splits must sum to 1.0"
    n_train = int(round(n_configs * train))
    n_val = int(round(n_configs * val))
    idx = np.arange(n_configs)
    return idx[:n_train], idx[n_train:n_train + n_val], idx[n_train + n_val:]


def silver_to_gold(silver_path: str | Path, gold_path: str | Path,
                    train: float = 0.8, val: float = 0.1, test: float = 0.1,
                    use_literature_split: Optional[bool] = None) -> Path:
    silver_path = Path(silver_path)
    gold_path = Path(gold_path)
    gold_path.parent.mkdir(parents=True, exist_ok=True)

    data = np.load(silver_path, allow_pickle=True)
    z, R, E, F = data["z"], data["R"], data["E"], data["F"]
    n_configs = R.shape[0]

    has_lit_split = bool(data["has_literature_split"]) if "has_literature_split" in data.files else False
    if use_literature_split is None:
        use_literature_split = has_lit_split

    if use_literature_split and has_lit_split:
        test_mask = data["literature_test_mask"]
        test_idx = np.where(test_mask)[0]
        remaining = np.where(~test_mask)[0]
        n_train_of_remaining = int(round(len(remaining) * (train / (train + val))))
        train_idx = remaining[:n_train_of_remaining]
        val_idx = remaining[n_train_of_remaining:]
    else:
        train_idx, val_idx, test_idx = trajectory_block_split(n_configs, train, val, test)

    assert set(train_idx).isdisjoint(test_idx), "train/test leakage detected"
    assert set(val_idx).isdisjoint(test_idx), "val/test leakage detected"
    assert set(train_idx).isdisjoint(val_idx), "train/val leakage detected"

    np.savez_compressed(
        gold_path, z=z, R=R, E=E, F=F,
        train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
        molecule=str(data["molecule"]), theory=str(data["theory"]),
        used_literature_split=bool(use_literature_split and has_lit_split),
    )
    return gold_path


PyTorch Geometric Dataset wrapper

In [48]:
class MD17Dataset(Dataset):
    def __init__(self, gold_path: str | Path, split: str = "train", cutoff_radius: float = 5.0):
        super().__init__()
        if split not in ("train", "val", "test"):
            raise ValueError(f"split must be train/val/test, got {split!r}")
        data = np.load(gold_path, allow_pickle=True)
        self.molecule = str(data["molecule"])
        self.theory = str(data["theory"])
        self.cutoff_radius = cutoff_radius
        self.split = split
        idx = data[f"{split}_idx"]
        self.z = torch.as_tensor(data["z"], dtype=torch.long)
        self.R = torch.as_tensor(data["R"][idx], dtype=torch.float32)
        self.E = torch.as_tensor(data["E"][idx], dtype=torch.float32)
        self.F = torch.as_tensor(data["F"][idx], dtype=torch.float32)

    def len(self) -> int:
        return self.R.shape[0]

    def get(self, idx: int) -> Data:
        pos = self.R[idx]
        edge_index, edge_attr = build_graph(pos.numpy(), cutoff=self.cutoff_radius)
        return Data(x=self.z.view(-1, 1), pos=pos, edge_index=edge_index, edge_attr=edge_attr,
                    y=self.E[idx].view(1), force=self.F[idx])

    def __repr__(self) -> str:
        return (f"MD17Dataset(molecule={self.molecule!r}, theory={self.theory!r}, "
                f"split={self.split!r}, n_configs={len(self)}, n_atoms={self.z.shape[0]})")


Pipeline driver

In [50]:
MD17_SOURCES = [
    ("md17_ethanol.npz", "ethanol", "dft", False),
    ("ethanol_ccsd_t.zip", "ethanol", "ccsd_t", True),
    ("benzene2018_dft.npz", "benzene", "dft", False),
    ("azobenzene_dft.npz", "azobenzene", "dft", False),
    ("paracetamol_dft.npz", "paracetamol", "dft", False),
    ("aspirin_ccsd.zip", "aspirin", "ccsd", True),
    ("malonaldehyde_ccsd_t.zip", "malonaldehyde", "ccsd_t", True),
    ("toluene_ccsd_t.zip", "toluene", "ccsd_t", True),
    ("md17_uracil.npz", "uracil", "dft", False),
]


def main() -> None:
    if not BRONZE.exists():
        print(f"No such directory: {BRONZE}")
        return

    SILVER.mkdir(parents=True, exist_ok=True)
    GOLD.mkdir(parents=True, exist_ok=True)

    print(f"\nBronze dir: {BRONZE}")
    print(f"{'Source':<28} {'Molecule':<14} {'Theory':<8} {'Status'}")
    print("-" * 78)

    results = []
    for filename, molecule, theory, use_lit_split in MD17_SOURCES:
        bronze_path = BRONZE / filename
        if not bronze_path.exists():
            print(f"{filename:<28} {molecule:<14} {theory:<8} MISSING (skip)")
            continue
        try:
            silver_path = bronze_to_silver(bronze_path, SILVER, molecule=molecule, theory=theory)
            gold_path = silver_to_gold(silver_path, GOLD / f"{molecule}_{theory}.npz", use_literature_split=use_lit_split)
            ds = MD17Dataset(gold_path, split="train")
            print(f"{filename:<28} {molecule:<14} {theory:<8} OK -> {gold_path.name} ({len(ds)} train configs, {ds.z.shape[0]} atoms)")
            results.append((molecule, theory, gold_path))
        except Exception as e:
            print(f"{filename:<28} {molecule:<14} {theory:<8} FAILED: {type(e).__name__}: {e}")

    print("-" * 78)
    print(f"{len(results)}/{len(MD17_SOURCES)} MD17 datasets processed into data/gold/md17/")

    if results:
        print("\nSample check — first successfully processed dataset:")
        molecule, theory, gold_path = results[0]
        ds = MD17Dataset(gold_path, split="train")
        item = ds[0]
        print(f"  {ds}")
        print(f"  Data(x={tuple(item.x.shape)}, pos={tuple(item.pos.shape)}, "
              f"edge_index={tuple(item.edge_index.shape)}, edge_attr={tuple(item.edge_attr.shape)}, "
              f"y={tuple(item.y.shape)}, force={tuple(item.force.shape)})")


if __name__ == "__main__":
    main()



Bronze dir: /content/HydrogenProductionMaterials_Discovery/data/bronze/md17
Source                       Molecule       Theory   Status
------------------------------------------------------------------------------
md17_ethanol.npz             ethanol        dft      OK -> ethanol_dft.npz (444074 train configs, 9 atoms)
ethanol_ccsd_t.zip           ethanol        ccsd_t   FAILED: UnpicklingError: Failed to interpret file <_io.BytesIO object at 0x7d8afd32f6f0> as a pickle
benzene2018_dft.npz          benzene        dft      OK -> benzene_dft.npz (39890 train configs, 12 atoms)
azobenzene_dft.npz           azobenzene     dft      OK -> azobenzene_dft.npz (79999 train configs, 24 atoms)
paracetamol_dft.npz          paracetamol    dft      OK -> paracetamol_dft.npz (85192 train configs, 20 atoms)
aspirin_ccsd.zip             aspirin        ccsd     FAILED: UnpicklingError: Failed to interpret file <_io.BytesIO object at 0x7d8afd1c83b0> as a pickle
malonaldehyde_ccsd_t.zip     malonaldehyd

Debugging the zip failure

In [51]:
import zipfile

path = "data/bronze/md17/aspirin_ccsd.zip"
with zipfile.ZipFile(path) as zf:
    for info in zf.infolist():
        print(info.filename, "-", info.file_size, "bytes")
    print()
    # peek at the first bytes of each member to identify the real format
    for info in zf.infolist():
        with zf.open(info.filename) as f:
            header = f.read(16)
        print(info.filename, "->", header)

aspirin_ccsd-test.npz - 487878 bytes
__MACOSX/ - 0 bytes
__MACOSX/._aspirin_ccsd-test.npz - 214 bytes
aspirin_ccsd-train.npz - 970664 bytes
__MACOSX/._aspirin_ccsd-train.npz - 214 bytes

aspirin_ccsd-test.npz -> b'PK\x03\x04\x14\x00\x00\x00\x08\x00\xc2\x81\x01M\x97g'
__MACOSX/ -> b''
__MACOSX/._aspirin_ccsd-test.npz -> b'\x00\x05\x16\x07\x00\x02\x00\x00Mac OS X'
aspirin_ccsd-train.npz -> b'PK\x03\x04\x14\x00\x00\x00\x08\x00\xc3\x81\x01M1f'
__MACOSX/._aspirin_ccsd-train.npz -> b'\x00\x05\x16\x07\x00\x02\x00\x00Mac OS X'
